# Module 9 — Dynamic Programming

This is the worked reference notebook: run live in lecture, fully solved.
The version students receive with TODOs in place of the solved parts is
`assignments/pds/a8-dynamic-programming/starter/dynamic_programming.py`.

## 1. Naive, memoized, and tabulated Fibonacci (Lecture 1, 2)

In [1]:
def fib(n):
    if n <= 1:
        return n
    return fib(n - 1) + fib(n - 2)

def fib_memo(n, cache=None):
    if cache is None:
        cache = {}
    if n in cache:
        return cache[n]
    if n <= 1:
        result = n
    else:
        result = fib_memo(n - 1, cache) + fib_memo(n - 2, cache)
    cache[n] = result
    return result

def fib_tabulated(n):
    if n <= 1:
        return n
    table = [0] * (n + 1)
    table[1] = 1
    for i in range(2, n + 1):
        table[i] = table[i - 1] + table[i - 2]
    return table[n]

assert fib(10) == fib_memo(10) == fib_tabulated(10) == 55
assert fib_memo(50) == fib_tabulated(50)   # fib(50) naive would take far too long
print("Fibonacci checks passed (naive, memoized, tabulated all agree)")

Fibonacci checks passed (naive, memoized, tabulated all agree)


## 2. 0/1 Knapsack, full table plus reconstruction (Lecture 2)

In [2]:
def knapsack(weights, values, W):
    n = len(weights)
    table = [[0] * (W + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for w in range(W + 1):
            if weights[i-1] > w:
                table[i][w] = table[i-1][w]
            else:
                table[i][w] = max(table[i-1][w], values[i-1] + table[i-1][w - weights[i-1]])
    return table

def knapsack_reconstruct(table, weights, n, W):
    chosen = []
    for i in range(n, 0, -1):
        if table[i][W] != table[i-1][W]:
            chosen.append(i - 1)
            W -= weights[i - 1]
    return chosen

weights = [1, 3, 4]
values = [15, 20, 30]
table = knapsack(weights, values, 4)
assert table[3][4] == 35   # matches Lecture 2's exact worked table
chosen = knapsack_reconstruct(table, weights, 3, 4)
assert sorted(chosen) == [0, 1]   # items 1 and 2 (0-indexed), matching Lecture 2's trace
print("Knapsack checks passed, matching Lecture 2's table and reconstruction exactly")

Knapsack checks passed, matching Lecture 2's table and reconstruction exactly


## 3. LCS and edit distance, with reconstruction (Lecture 3)

In [3]:
def lcs_table(s1, s2):
    n, m = len(s1), len(s2)
    L = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if s1[i-1] == s2[j-1]:
                L[i][j] = L[i-1][j-1] + 1
            else:
                L[i][j] = max(L[i-1][j], L[i][j-1])
    return L

def lcs_reconstruct(table, s1, s2, i, j):
    if i == 0 or j == 0:
        return ""
    if s1[i-1] == s2[j-1]:
        return lcs_reconstruct(table, s1, s2, i-1, j-1) + s1[i-1]
    elif table[i-1][j] >= table[i][j-1]:
        return lcs_reconstruct(table, s1, s2, i-1, j)
    else:
        return lcs_reconstruct(table, s1, s2, i, j-1)

L = lcs_table("ABC", "AC")
assert L[3][2] == 2   # matches Lecture 3's exact table
assert lcs_reconstruct(L, "ABC", "AC", 3, 2) == "AC"

def edit_distance(s1, s2):
    n, m = len(s1), len(s2)
    D = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1): D[i][0] = i
    for j in range(m + 1): D[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if s1[i-1] == s2[j-1]:
                D[i][j] = D[i-1][j-1]
            else:
                D[i][j] = 1 + min(D[i-1][j-1], D[i-1][j], D[i][j-1])
    return D[n][m]

assert edit_distance("kitten", "sitting") == 3
print("LCS and edit distance checks passed, matching Lecture 3 exactly")

LCS and edit distance checks passed, matching Lecture 3 exactly


## 4. Coin change, with reconstruction, and the greedy counterexample (Lecture 4)

In [4]:
def coin_change_reconstruct(coins, target):
    f = [0] * (target + 1)
    choice = [None] * (target + 1)
    for a in range(1, target + 1):
        best = float('inf')
        for c in coins:
            if c <= a and 1 + f[a - c] < best:
                best = 1 + f[a - c]
                choice[a] = c
        f[a] = best
    result, a = [], target
    while a > 0:
        result.append(choice[a])
        a -= choice[a]
    return f[target], result

def greedy_coin_change(coins, target):
    coins = sorted(coins, reverse=True)
    count = 0
    for c in coins:
        count += target // c
        target %= c
    return count

value, coins_used = coin_change_reconstruct([1, 3, 4], 6)
assert value == 2 and sorted(coins_used) == [3, 3]   # matches Lecture 4's exact trace

# The greedy counterexample from Lecture 4
assert greedy_coin_change([1, 3, 4], 6) == 3   # greedy: 4+1+1
assert value == 2   # DP proves the true optimum is 2 — greedy is provably wrong here

# Standard currency: greedy happens to match DP for every target 0-100
for t in range(101):
    dp_value, _ = coin_change_reconstruct([1, 5, 10, 25], t) if t > 0 else (0, [])
    assert dp_value == greedy_coin_change([1, 5, 10, 25], t)
print("Coin change checks passed, greedy counterexample confirmed, standard currency confirmed safe")

Coin change checks passed, greedy counterexample confirmed, standard currency confirmed safe
